# BÀI TẬP VỀ NHÀ: MÔ PHỎNG MÁY HÚT BỤI BẰNG BFS VÀ DFS

Chào mừng bạn đến với mô phỏng trực quan **Thế giới Máy hút bụi (Vacuum-cleaner World)** ứng dụng thuật toán tìm kiếm **BFS (Breadth-First Search)** và **DFS (Depth-First Search)**.

## 1. Giới thiệu thuật toán
- **BFS (Tìm kiếm theo chiều rộng)**: Sử dụng cấu trúc dữ liệu **Hàng đợi (Queue - FIFO)**. BFS mở rộng các ô theo dạng làn sóng vòng tròn đồng tâm xuất phát từ máy hút. Điều này đảm bảo tìm ra **đường đi ngắn nhất (tối ưu nhất)** đến ô bụi gần nhất, nhưng tốn nhiều bộ nhớ hơn để lưu trữ hàng đợi.
- **DFS (Tìm kiếm theo chiều sâu)**: Sử dụng cấu trúc dữ liệu **Ngăn xếp (Stack - LIFO)**. DFS đâm sâu tối đa theo một hướng trước khi quay lui (backtrack). Nó tốn ít bộ nhớ hơn nhưng **không đảm bảo đường đi ngắn nhất** (máy có thể đi vòng vèo qua nhiều ô trước khi chạm tới đống bụi ngay sát cạnh).

## 2. Các chức năng của giao diện đồ họa (Tkinter):
1. **Bản đồ lưới 5x5 tương tác**: Bạn có thể chuyển sang công cụ `Vẽ Bụi` để click thêm/bớt bụi, hoặc công cụ `Đặt Máy hút` để di chuyển vị trí ban đầu của máy hút bụi trên lưới.
2. **Lựa chọn thuật toán**: Cho phép chọn giữa BFS và DFS để chạy mô phỏng đối chiếu.
3. **Thanh trượt tốc độ (Speed Slider)**: Điều chỉnh độ trễ giữa mỗi bước lập kế hoạch/di chuyển của máy hút từ 50ms (cực nhanh) đến 1500ms (chậm để dễ quan sát).
4. **Biên tìm kiếm động (Frontier Tracker)**: Thanh bên hiển thị trực tiếp danh sách các đường đi đang nằm trong hàng đợi FIFO (nếu chọn BFS) hoặc ngăn xếp LIFO (nếu chọn DFS) thay đổi theo từng bước duyệt.
5. **Nhật ký giải thích từng bước**: Bản ghi thuyết minh tự nhiên giải thích lý do tại sao thuật toán lại chọn duyệt ô đó, thêm lân cận nào vào biên, và khi nào tìm thấy đích.
6. **Bảng so sánh hiệu năng**: Ghi lại lịch sử chạy mô phỏng của bạn để trực quan đối chiếu: Tổng số ô đã duyệt, lượng bộ nhớ Frontier tối đa, độ dài đường đi của cả BFS và DFS.

## 3. Cách chạy ứng dụng
Chạy ô code dưới đây để mở cửa sổ mô phỏng Tkinter!

In [3]:
import tkinter as tk
from tkinter import ttk, messagebox
import random
import time
from collections import deque

class VacuumSimulator:
    def __init__(self):
        # Grid state: 5x5 grid
        self.grid = [[0 for _ in range(5)] for _ in range(5)]
        
        # Default dirt locations
        self.grid[1][2] = 1
        self.grid[3][1] = 1
        self.grid[4][3] = 1
        
        # Vacuum state
        self.vacuum_pos = (0, 0)
        self.state = "idle"  # idle, planning, moving, finished
        
        # Search variables
        self.frontier = deque()
        self.visited = set()
        self.current_exploring = None
        self.final_path = []
        self.move_index = 0
        self.current_algo = "BFS"
        
        # Search stats
        self.nodes_expanded = 0
        self.max_frontier_size = 0
        self.dirt_cleaned = 0

    def toggle_dirt(self, r, c):
        if 0 <= r < 5 and 0 <= c < 5:
            # Cannot place dirt on the vacuum cleaner position
            if (r, c) == self.vacuum_pos:
                return
            self.grid[r][c] = 1 - self.grid[r][c]

    def set_vacuum(self, r, c):
        if 0 <= r < 5 and 0 <= c < 5:
            self.vacuum_pos = (r, c)
            self.grid[r][c] = 0  # Clean dirt under the vacuum instantly when placed

    def has_dirt(self):
        for r in range(5):
            for c in range(5):
                if self.grid[r][c] == 1:
                    return True
        return False

    def reset_search(self):
        self.state = "idle"
        self.frontier.clear()
        self.visited.clear()
        self.current_exploring = None
        self.final_path = []
        self.move_index = 0

    def reset_grid(self):
        self.grid = [[0 for _ in range(5)] for _ in range(5)]
        self.vacuum_pos = (0, 0)
        self.reset_search()

    def start_planning(self, algo):
        self.state = "planning"
        self.current_algo = algo
        self.visited = set()
        self.frontier.clear()
        
        # Start path is just the vacuum position
        self.frontier.append([self.vacuum_pos])
        
        self.current_exploring = None
        self.final_path = []
        self.move_index = 0
        
        # Reset stats
        self.nodes_expanded = 0
        self.max_frontier_size = len(self.frontier)
        
        return f"Bắt đầu lập kế hoạch bằng {algo} tìm đường đến bụi gần nhất..."

    def step_search(self, algo):
        if self.state != "planning":
            return ""
            
        if not self.frontier:
            self.state = "idle"
            return "Không tìm thấy đường đi đến bụi nào! Lập kế hoạch thất bại."
            
        # Pop from frontier based on algorithm
        if algo == "BFS":
            path = self.frontier.popleft()
            log_msg = f"BFS [FIFO Queue]: Lấy đường đi đầu Hàng đợi: {path}"
        else:  # DFS
            path = self.frontier.pop()
            log_msg = f"DFS [LIFO Stack]: Lấy đường đi đỉnh Ngăn xếp: {path}"
            
        current = path[-1]
        self.current_exploring = current
        
        # DFS cycle/visited check
        if algo == "DFS" and current in self.visited:
            return f"DFS: Ô {current} đã duyệt trước đó. Bỏ qua để tránh lặp vòng."
            
        # Goal Test
        if self.grid[current[0]][current[1]] == 1:
            self.final_path = path
            self.state = "moving"
            self.move_index = 0
            return f"ĐÍCH ĐẾN! Đã tìm thấy Bụi tại ô {current}!"
            
        self.visited.add(current)
        self.nodes_expanded += 1
        
        # Explore neighbors
        r, c = current
        neighbors = []
        directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # UP, RIGHT, DOWN, LEFT
        dir_names = ["Lên", "Phải", "Xuống", "Trái"]
        
        for i, (dr, dc) in enumerate(directions):
            nr, nc = r + dr, c + dc
            if 0 <= nr < 5 and 0 <= nc < 5:
                if (nr, nc) not in self.visited:
                    # BFS optimization: skip duplicate states inside queue
                    already_in_frontier = False
                    if algo == "BFS":
                        for p in self.frontier:
                            if p[-1] == (nr, nc):
                                already_in_frontier = True
                                break
                                
                    if not already_in_frontier:
                        new_path = list(path) + [(nr, nc)]
                        neighbors.append((new_path, dir_names[i]))
                        
        # Explored neighbors handling
        if algo == "DFS":
            # Push in reverse priority order so UP is explored first (LIFO order)
            for new_path, name in reversed(neighbors):
                self.frontier.append(new_path)
                log_msg += f"\nDFS: Đẩy {new_path[-1]} (hướng {name}) vào đỉnh Ngăn xếp (LIFO)"
        else:  # BFS
            for new_path, name in neighbors:
                self.frontier.append(new_path)
                log_msg += f"\nBFS: Đẩy {new_path[-1]} (hướng {name}) vào cuối Hàng đợi (FIFO)"
                
        # Memory track
        self.max_frontier_size = max(self.max_frontier_size, len(self.frontier))
        
        return log_msg

    def step_move(self):
        if self.state != "moving":
            return ""
            
        if self.move_index < len(self.final_path):
            self.vacuum_pos = self.final_path[self.move_index]
            log_msg = f"MÁY HÚT: Di chuyển đến ô {self.vacuum_pos}"
            self.move_index += 1
            return log_msg
        else:
            # SUCK action
            r, c = self.vacuum_pos
            if self.grid[r][c] == 1:
                self.grid[r][c] = 0
                self.dirt_cleaned += 1
                log_msg = f"MÁY HÚT [HÚT]: Hút bụi tại ô ({r},{c}) thành công!"
            else:
                log_msg = f"MÁY HÚT: Không có bụi tại ô ({r},{c}) để hút!"
                
            # Check for remaining dirts
            if self.has_dirt():
                log_msg += "\nBản đồ vẫn còn bụi! Tiếp tục lập kế hoạch tìm ô bụi mới..."
                self.start_planning(self.current_algo)
            else:
                log_msg += "\nHOÀN THÀNH! Đã hút sạch toàn bộ bụi bẩn trên bản đồ!"
                self.state = "finished"
                
            return log_msg


class VacuumSimulatorApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Mô Phỏng Máy Hút Bụi AI - BFS vs DFS")
        self.root.geometry("1150x750")
        self.root.configure(bg="#1a1b26")
        
        # Instantiate Core TDD Model
        self.sim = VacuumSimulator()
        
        # Simulator GUI state
        self.auto_run = False
        self.active_tool = "dirt"  # dirt, vacuum
        
        # Create UI
        self.create_styles()
        self.create_widgets()
        self.draw_grid()
        self.update_stats()
        self.log("Ứng dụng đã sẵn sàng! Click chuột lên lưới 5x5 để thêm Bụi hoặc đặt Máy hút bụi.")
        
    def create_styles(self):
        style = ttk.Style()
        style.theme_use("clam")
        
        # Configure colors
        style.configure("TFrame", background="#1a1b26")
        style.configure("TLabel", background="#1a1b26", foreground="#a9b1d6", font=("Segoe UI", 10))
        style.configure("Header.TLabel", background="#1a1b26", foreground="#c0caf5", font=("Segoe UI", 16, "bold"))
        style.configure("SubHeader.TLabel", background="#1a1b26", foreground="#7aa2f7", font=("Segoe UI", 12, "bold"))
        
        # Configure Radiobuttons
        style.configure("TRadiobutton", background="#1a1b26", foreground="#a9b1d6", font=("Segoe UI", 10))
        
        # Flat custom buttons (Modern Tokyo Night Palette)
        style.configure("Action.TButton", font=("Segoe UI", 10, "bold"), background="#7aa2f7", foreground="#16161e")
        style.map("Action.TButton", background=[("active", "#89ddff")])
        
        style.configure("Reset.TButton", font=("Segoe UI", 10, "bold"), background="#f7768e", foreground="#16161e")
        style.map("Reset.TButton", background=[("active", "#ff9e64")])
        
    def create_widgets(self):
        # Main layout frame
        main_layout = ttk.Frame(self.root, padding=15)
        main_layout.pack(fill=tk.BOTH, expand=True)
        
        # Left Panel (Grid and Legend)
        left_panel = ttk.Frame(main_layout)
        left_panel.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 15))
        
        # Header title
        title_label = ttk.Label(left_panel, text="MÁY HÚT BỤI THÔNG MINH - AI SEARCH SIMULATOR", style="Header.TLabel")
        title_label.pack(anchor=tk.W, pady=(0, 10))
        
        # Canvas Frame
        canvas_frame = ttk.Frame(left_panel)
        canvas_frame.pack(pady=5)
        
        self.canvas = tk.Canvas(
            canvas_frame, 
            width=450, 
            height=450, 
            bg="#16161e", 
            highlightthickness=2, 
            highlightbackground="#383e56"
        )
        self.canvas.pack()
        self.canvas.bind("<Button-1>", self.on_canvas_click)
        
        # Legend Frame
        legend_frame = ttk.Frame(left_panel, padding=5)
        legend_frame.pack(fill=tk.X, pady=10)
        
        legends = [
            ("Máy hút", "#ff7c3b", "circle"),
            ("Bụi bẩn", "#ff5d5d", "dots"),
            ("Đang xét", "#00f2fe", "rect"),
            ("Trong biên", "#3b82f6", "border"),
            ("Đã duyệt", "#2e3047", "rect"),
            ("Đường đi", "#10b981", "line")
        ]
        
        for i, (name, color, type_) in enumerate(legends):
            item_frame = ttk.Frame(legend_frame)
            item_frame.pack(side=tk.LEFT, expand=True, padx=2)
            
            mini_canvas = tk.Canvas(item_frame, width=24, height=24, bg="#16161e", highlightthickness=1, highlightbackground="#383e56")
            mini_canvas.pack(side=tk.LEFT, padx=3)
            
            if type_ == "circle":
                mini_canvas.create_oval(3, 3, 21, 21, fill=color, outline="#16161e")
            elif type_ == "dots":
                mini_canvas.create_oval(6, 6, 18, 18, fill=color, outline="")
            elif type_ == "rect":
                mini_canvas.create_rectangle(3, 3, 21, 21, fill=color, outline="#383e56")
            elif type_ == "border":
                mini_canvas.create_rectangle(3, 3, 21, 21, fill="#16161e", outline=color, width=2)
            elif type_ == "line":
                mini_canvas.create_line(3, 12, 21, 12, fill=color, width=3, arrow=tk.LAST)
                
            lbl = ttk.Label(item_frame, text=name, font=("Segoe UI", 9))
            lbl.pack(side=tk.LEFT)
            
        # Tool Frame
        tool_frame = ttk.Frame(left_panel)
        tool_frame.pack(fill=tk.X, pady=5)
        
        ttk.Label(tool_frame, text="Công cụ vẽ lưới:", font=("Segoe UI", 10, "bold")).pack(side=tk.LEFT, padx=(0, 10))
        
        self.tool_var = tk.StringVar(value="dirt")
        
        btn_tool_dirt = ttk.Radiobutton(tool_frame, text="Vẽ Bụi", variable=self.tool_var, value="dirt")
        btn_tool_dirt.pack(side=tk.LEFT, padx=10)
        
        btn_tool_vac = ttk.Radiobutton(tool_frame, text="Đặt Máy hút", variable=self.tool_var, value="vacuum")
        btn_tool_vac.pack(side=tk.LEFT, padx=10)
        
        # Right Panel (Controls, Tracker, Log, History)
        right_panel = ttk.Frame(main_layout)
        right_panel.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)
        
        # Tabbed system on the right to organize space elegantly
        tab_control = ttk.Notebook(right_panel)
        tab_control.pack(fill=tk.BOTH, expand=True)
        
        # Tab 1: Control & Data Tracker
        tab_control_frame = ttk.Frame(tab_control, padding=10)
        tab_control.add(tab_control_frame, text=" Điều khiển & Biên ")
        
        # Config & Selection inside Tab 1
        config_frame = ttk.Frame(tab_control_frame)
        config_frame.pack(fill=tk.X, pady=(0, 10))
        
        ttk.Label(config_frame, text="Thuật toán:", style="SubHeader.TLabel").pack(anchor=tk.W, pady=(0, 5))
        
        self.algo_var = tk.StringVar(value="BFS")
        btn_bfs = ttk.Radiobutton(config_frame, text="BFS (Hàng đợi - Rộng)", variable=self.algo_var, value="BFS", command=self.on_algo_change)
        btn_bfs.pack(anchor=tk.W, padx=10, pady=2)
        
        btn_dfs = ttk.Radiobutton(config_frame, text="DFS (Ngăn xếp - Sâu)", variable=self.algo_var, value="DFS", command=self.on_algo_change)
        btn_dfs.pack(anchor=tk.W, padx=10, pady=2)
        
        # Speed slider
        speed_frame = ttk.Frame(tab_control_frame)
        speed_frame.pack(fill=tk.X, pady=5)
        ttk.Label(speed_frame, text="Độ trễ bước mô phỏng (ms):", font=("Segoe UI", 10, "bold")).pack(anchor=tk.W)
        self.speed_slider = tk.Scale(speed_frame, from_=50, to_=1500, orient=tk.HORIZONTAL, bg="#1a1b26", fg="#a9b1d6", highlightthickness=0, activebackground="#7aa2f7")
        self.speed_slider.set(300)
        self.speed_slider.pack(fill=tk.X, padx=10, pady=5)
        
        # Buttons panel
        btn_panel = ttk.Frame(tab_control_frame)
        btn_panel.pack(fill=tk.X, pady=10)
        
        self.btn_start = ttk.Button(btn_panel, text="Bắt đầu", style="Action.TButton", command=self.on_btn_start)
        self.btn_start.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        
        btn_step = ttk.Button(btn_panel, text="Từng bước", command=self.on_btn_step)
        btn_step.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        
        btn_rand = ttk.Button(btn_panel, text="Ngẫu nhiên", command=self.on_btn_random)
        btn_rand.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        
        btn_reset = ttk.Button(btn_panel, text="Reset Lưới", style="Reset.TButton", command=self.on_btn_reset_grid)
        btn_reset.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        
        # Frontier list visualizer
        ttk.Label(tab_control_frame, text="Cấu trúc dữ liệu Frontier (Biên tìm kiếm):", font=("Segoe UI", 10, "bold")).pack(anchor=tk.W, pady=(10, 5))
        
        frontier_frame = ttk.Frame(tab_control_frame)
        frontier_frame.pack(fill=tk.BOTH, expand=True, pady=5)
        
        self.frontier_listbox = tk.Listbox(
            frontier_frame, 
            bg="#16161e", 
            fg="#9ece6a", 
            selectbackground="#2f3549", 
            selectforeground="#c0caf5",
            font=("Consolas", 10),
            highlightcolor="#7aa2f7",
            borderwidth=1,
            relief=tk.FLAT
        )
        self.frontier_listbox.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        scr_frontier = ttk.Scrollbar(frontier_frame, orient=tk.VERTICAL, command=self.frontier_listbox.yview)
        scr_frontier.pack(side=tk.RIGHT, fill=tk.Y)
        self.frontier_listbox.config(yscrollcommand=scr_frontier.set)
        
        # Live Stats Frame
        stats_frame = ttk.LabelFrame(tab_control_frame, text=" Số liệu chạy hiện tại ", padding=10)
        stats_frame.pack(fill=tk.X, pady=10)
        
        self.lbl_expanded = ttk.Label(stats_frame, text="Ô đã duyệt (Expanded): 0")
        self.lbl_expanded.pack(anchor=tk.W, pady=2)
        
        self.lbl_max_frontier = ttk.Label(stats_frame, text="Bộ nhớ tối đa (Max Frontier): 0")
        self.lbl_max_frontier.pack(anchor=tk.W, pady=2)
        
        self.lbl_path_len = ttk.Label(stats_frame, text="Độ dài đường đi: 0 bước")
        self.lbl_path_len.pack(anchor=tk.W, pady=2)
        
        self.lbl_cleaned = ttk.Label(stats_frame, text="Bụi đã hút: 0")
        self.lbl_cleaned.pack(anchor=tk.W, pady=2)
        
        # Tab 2: Explanation Log
        tab_log_frame = ttk.Frame(tab_control, padding=10)
        tab_control.add(tab_log_frame, text=" Nhật ký giải thích ")
        
        ttk.Label(tab_log_frame, text="Chi tiết các bước ra quyết định của thuật toán:", font=("Segoe UI", 10, "bold")).pack(anchor=tk.W, pady=(0, 5))
        
        log_txt_frame = ttk.Frame(tab_log_frame)
        log_txt_frame.pack(fill=tk.BOTH, expand=True)
        
        self.log_text = tk.Text(
            log_txt_frame,
            bg="#16161e",
            fg="#a9b1d6",
            font=("Consolas", 10),
            state=tk.DISABLED,
            wrap=tk.WORD,
            borderwidth=1,
            relief=tk.FLAT
        )
        self.log_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        scr_log = ttk.Scrollbar(log_txt_frame, orient=tk.VERTICAL, command=self.log_text.yview)
        scr_log.pack(side=tk.RIGHT, fill=tk.Y)
        self.log_text.config(yscrollcommand=scr_log.set)
        
        # Tab 3: History & Comparison
        tab_hist_frame = ttk.Frame(tab_control, padding=10)
        tab_control.add(tab_hist_frame, text=" So sánh hiệu năng ")
        
        ttk.Label(tab_hist_frame, text="Bảng đối chiếu hiệu năng BFS vs DFS:", style="SubHeader.TLabel").pack(anchor=tk.W, pady=(0, 5))
        ttk.Label(tab_hist_frame, text="Hãy chạy cả BFS và DFS trên cùng một cách sắp xếp Bụi để thấy rõ sự khác biệt!").pack(anchor=tk.W, pady=(0, 10))
        
        # Treeview table
        columns = ("algo", "dirts", "expanded", "max_frontier", "path_len")
        self.history_tree = ttk.Treeview(tab_hist_frame, columns=columns, show="headings", height=10)
        self.history_tree.pack(fill=tk.BOTH, expand=True)
        
        self.history_tree.heading("algo", text="Thuật toán")
        self.history_tree.heading("dirts", text="Tổng số Bụi")
        self.history_tree.heading("expanded", text="Ô đã duyệt")
        self.history_tree.heading("max_frontier", text="Bộ nhớ Max (Biên)")
        self.history_tree.heading("path_len", text="Độ dài đường đi")
        
        self.history_tree.column("algo", width=100, anchor=tk.CENTER)
        self.history_tree.column("dirts", width=80, anchor=tk.CENTER)
        self.history_tree.column("expanded", width=100, anchor=tk.CENTER)
        self.history_tree.column("max_frontier", width=120, anchor=tk.CENTER)
        self.history_tree.column("path_len", width=120, anchor=tk.CENTER)
        
        btn_clear_hist = ttk.Button(tab_hist_frame, text="Xóa lịch sử so sánh", command=self.on_clear_history)
        btn_clear_hist.pack(anchor=tk.E, pady=10)
        
    def log(self, msg):
        self.log_text.config(state=tk.NORMAL)
        timestamp = time.strftime("%H:%M:%S")
        self.log_text.insert(tk.END, f"[{timestamp}] {msg}\n")
        self.log_text.see(tk.END)
        self.log_text.config(state=tk.DISABLED)
        
    def draw_grid(self):
        self.canvas.delete("all")
        cell_size = 90  # 450 / 5
        
        for r in range(5):
            for c in range(5):
                x1, y1 = c * cell_size, r * cell_size
                x2, y2 = x1 + cell_size, y1 + cell_size
                
                # Default cell background (Refined Slate Blue tint)
                bg_color = "#24283b"
                border_color = "#383e56"
                
                # Color code cell based on search states - Harmonious transition
                if (r, c) == self.sim.current_exploring:
                    bg_color = "#2e3f42"  # Slate Blue-Teal highlight for exploring node
                elif (r, c) in [p[-1] for p in self.sim.frontier]:
                    bg_color = "#202c40"  # Soft tech blue tint for frontier
                elif (r, c) in self.sim.visited:
                    bg_color = "#2b2d3d"  # Soft slate-grey for visited (Quiet history)
                    border_color = "#343b58"
                    
                # Highlight cells along the final path
                if (r, c) in self.sim.final_path:
                    bg_color = "#1b2f2b"  # Clean, quiet green tint for final path cells
                    
                self.canvas.create_rectangle(x1, y1, x2, y2, fill=bg_color, outline=border_color, width=1)
                
                # Outer border for frontier nodes
                if (r, c) in [p[-1] for p in self.sim.frontier]:
                    self.canvas.create_rectangle(x1+4, y1+4, x2-4, y2-4, outline="#3b82f6", width=2)
                    
                # Active glowing border for current expanding node
                if (r, c) == self.sim.current_exploring:
                    self.canvas.create_rectangle(x1+3, y1+3, x2-3, y2-3, outline="#00f2fe", width=2)
                
                # Draw coordinate text (small, muted)
                self.canvas.create_text(x1 + 18, y1 + 15, text=f"{r},{c}", fill="#565f89", font=("Consolas", 8))
                
                # Draw elegant single-circle dirt with modern details
                if self.sim.grid[r][c] == 1:
                    cx = x1 + cell_size / 2
                    cy = y1 + cell_size / 2
                    self.canvas.create_oval(cx-15, cy-15, cx+15, cy+15, fill="#ff5d5d", outline="#16161e", width=1)
                    # Accent inner core
                    self.canvas.create_oval(cx-6, cy-6, cx+6, cy+6, fill="#9e2a2b", outline="")
                    # Clean label
                    self.canvas.create_text(cx, cy + 26, text="BỤI", fill="#ff5d5d", font=("Segoe UI", 7, "bold"))
                    
        # Draw path line if final path exists
        if len(self.sim.final_path) > 1:
            points = []
            for r, c in self.sim.final_path:
                cx = c * cell_size + cell_size / 2
                cy = r * cell_size + cell_size / 2
                points.append((cx, cy))
            for i in range(len(points)-1):
                self.canvas.create_line(points[i][0], points[i][1], points[i+1][0], points[i+1][1], fill="#10b981", width=3, arrow=tk.LAST)
                
        # Draw Vacuum cleaner (Clean, white body with orange highlights)
        vr, vc = self.sim.vacuum_pos
        vx = vc * cell_size + cell_size / 2
        vy = vr * cell_size + cell_size / 2
        r_vac = 25
        
        # Pure white circular body
        self.canvas.create_oval(vx-r_vac, vy-r_vac, vx+r_vac, vy+r_vac, fill="#ffffff", outline="#16161e", width=2)
        # Inner orange ring
        self.canvas.create_oval(vx-18, vy-18, vx+18, vy+18, fill="#ff7c3b", outline="")
        # Core black body
        self.canvas.create_oval(vx-10, vy-10, vx+10, vy+10, fill="#16161e", outline="")
        
        # Glowing LED status center
        led_color = "#ff7c3b"  # idle = orange
        if self.sim.state == "planning":
            led_color = "#00f2fe"  # planning = cyan
        elif self.sim.state == "moving":
            led_color = "#3b82f6"  # moving = blue
        elif self.sim.state == "finished":
            led_color = "#10b981"  # finished = emerald green
            
        self.canvas.create_oval(vx-4, vy-4, vx+4, vy+4, fill=led_color, outline="")
        
        # Muted orientation line
        self.canvas.create_line(vx-r_vac, vy, vx+r_vac, vy, fill="#16161e", width=1)
        self.canvas.create_text(vx, vy - 35, text="MÁY HÚT", fill="#ffffff", font=("Segoe UI", 9, "bold"))
        
    def on_canvas_click(self, event):
        if self.sim.state in ["planning", "moving"]:
            messagebox.showwarning("Cảnh báo", "Không thể sửa bản đồ khi mô phỏng đang chạy!")
            return
            
        cell_size = 90
        c = event.x // cell_size
        r = event.y // cell_size
        
        if 0 <= r < 5 and 0 <= c < 5:
            if self.tool_var.get() == "vacuum":
                self.sim.grid[r][c] = 0  # no dirt under vacuum
                self.sim.vacuum_pos = (r, c)
                self.log(f"Đã đặt Máy hút bụi tại ({r}, {c})")
            else:
                if (r, c) == self.sim.vacuum_pos:
                    messagebox.showwarning("Cảnh báo", "Không thể vẽ bụi đè lên vị trí máy hút bụi!")
                    return
                self.sim.toggle_dirt(r, c)
                state_str = "Thêm bụi" if self.sim.grid[r][c] == 1 else "Xóa bụi"
                self.log(f"Đã {state_str} tại ô ({r}, {c})")
                
            self.sim.reset_search()
            self.update_stats()
            self.update_frontier_ui()
            self.draw_grid()
            
    def on_algo_change(self):
        self.sim.reset_search()
        self.update_stats()
        self.update_frontier_ui()
        self.draw_grid()
        
    def update_stats(self):
        path_length = len(self.sim.final_path) - 1 if self.sim.final_path else 0
        self.lbl_expanded.config(text=f"Ô đã duyệt (Expanded): {self.sim.nodes_expanded}")
        self.lbl_max_frontier.config(text=f"Bộ nhớ tối đa (Max Frontier): {self.sim.max_frontier_size}")
        self.lbl_path_len.config(text=f"Độ dài đường đi: {path_length} bước")
        self.lbl_cleaned.config(text=f"Bụi đã hút: {self.sim.dirt_cleaned}")
        
    def update_frontier_ui(self):
        self.frontier_listbox.delete(0, tk.END)
        for path in self.sim.frontier:
            coord_str = " -> ".join([f"({r},{c})" for r, c in path])
            self.frontier_listbox.insert(tk.END, coord_str)
            
    def on_btn_random(self):
        if self.sim.state in ["planning", "moving"]:
            return
        self.sim.grid = [[0 for _ in range(5)] for _ in range(5)]
        num_dirt = random.randint(3, 6)
        count = 0
        while count < num_dirt:
            r = random.randint(0, 4)
            c = random.randint(0, 4)
            if (r, c) != self.sim.vacuum_pos and self.sim.grid[r][c] == 0:
                self.sim.grid[r][c] = 1
                count += 1
        self.sim.reset_search()
        self.draw_grid()
        self.update_stats()
        self.update_frontier_ui()
        self.log(f"Đã tạo ngẫu nhiên {num_dirt} ô bụi mới trên lưới.")
        
    def on_btn_reset_grid(self):
        self.auto_run = False
        self.btn_start.config(text="Bắt đầu")
        self.sim.reset_grid()
        self.draw_grid()
        self.update_stats()
        self.update_frontier_ui()
        self.log("Đã làm sạch toàn bộ lưới và đặt Máy hút về góc (0, 0).")
        
    def on_clear_history(self):
        for item in self.history_tree.get_children():
            self.history_tree.delete(item)
            
    def on_btn_start(self):
        if self.sim.state in ["planning", "moving"]:
            # Pause simulation
            self.auto_run = False
            self.sim.state = "idle"
            self.btn_start.config(text="Tiếp tục")
            self.log("Đã tạm dừng mô phỏng.")
        else:
            # Start/Resume
            if not self.sim.has_dirt():
                messagebox.showwarning("Cảnh báo", "Vui lòng vẽ bụi lên lưới trước khi bắt đầu!")
                return
                
            self.auto_run = True
            self.btn_start.config(text="Tạm dừng")
            
            if self.sim.state == "finished" or self.sim.state == "idle":
                # Fresh start or resume
                if not self.sim.frontier and not self.sim.final_path:
                    self.sim.nodes_expanded = 0
                    self.sim.max_frontier_size = 0
                    self.sim.dirt_cleaned = 0
                    
                    algo = self.algo_var.get()
                    log_msg = self.sim.start_planning(algo)
                    self.log(log_msg)
                    
                    self.update_stats()
                    self.update_frontier_ui()
                    self.draw_grid()
                    
                    self.root.after(self.speed_slider.get(), self.step_search)
                else:
                    # Resume from where we were
                    if self.sim.final_path:
                        self.sim.state = "moving"
                        self.step_move()
                    else:
                        self.sim.state = "planning"
                        self.sim.state = "planning"
                        self.step_search()
                        
    def step_search(self):
        if self.sim.state != "planning":
            return
            
        algo = self.algo_var.get()
        log_msg = self.sim.step_search(algo)
        self.log(log_msg)
        
        self.update_stats()
        self.update_frontier_ui()
        self.draw_grid()
        
        if self.sim.state == "moving":
            if self.auto_run:
                self.root.after(self.speed_slider.get(), self.step_move)
        elif self.sim.state == "idle":
            self.btn_start.config(text="Bắt đầu")
            self.auto_run = False
            messagebox.showerror("Lỗi", "Không tìm thấy đường đi khả thi đến ô bụi nào!")
        else:
            if self.auto_run:
                self.root.after(self.speed_slider.get(), self.step_search)
                
    def step_move(self):
        if self.sim.state != "moving":
            return
            
        log_msg = self.sim.step_move()
        self.log(log_msg)
        
        self.update_stats()
        self.update_frontier_ui()
        self.draw_grid()
        
        if self.sim.state == "planning":
            if self.auto_run:
                self.root.after(self.speed_slider.get(), self.step_search)
        elif self.sim.state == "finished":
            self.btn_start.config(text="Bắt đầu")
            self.auto_run = False
            self.save_history()
            messagebox.showinfo(
                "Hoàn thành", 
                f"Đã dọn sạch lưới bằng {self.algo_var.get()}!\n"
                f"- Tổng ô đã duyệt: {self.sim.nodes_expanded}\n"
                f"- Bộ nhớ Frontier tối đa: {self.sim.max_frontier_size} phần tử"
            )
        else:
            if self.auto_run:
                self.root.after(self.speed_slider.get(), self.step_move)
                
    def on_btn_step(self):
        if self.sim.state in ["planning", "moving"]:
            # Pause first
            self.auto_run = False
            self.sim.state = "idle"
            self.btn_start.config(text="Tiếp tục")
            self.log("Đã chuyển sang chế độ thủ công từng bước.")
            return
            
        if not self.sim.has_dirt():
            messagebox.showwarning("Cảnh báo", "Vui lòng vẽ bụi lên lưới trước!")
            return
            
        self.auto_run = False
        self.btn_start.config(text="Tiếp tục")
        
        if self.sim.state == "finished" or self.sim.state == "idle":
            if not self.sim.frontier and not self.sim.final_path:
                self.sim.nodes_expanded = 0
                self.sim.max_frontier_size = 0
                self.sim.dirt_cleaned = 0
                
                algo = self.algo_var.get()
                log_msg = self.sim.start_planning(algo)
                self.log(log_msg)
                
                self.update_stats()
                self.update_frontier_ui()
                self.draw_grid()
                
        # Perform exactly 1 step
        if self.sim.final_path:
            self.sim.state = "moving"
            self.step_move()
        else:
            self.sim.state = "planning"
            self.step_search()
            
    def save_history(self):
        algo = self.algo_var.get()
        total_dirt = self.sim.dirt_cleaned
        path_length = len(self.sim.final_path) - 1 if self.sim.final_path else 0
        self.history_tree.insert(
            "", 
            tk.END, 
            values=(algo, total_dirt, self.sim.nodes_expanded, self.sim.max_frontier_size, f"{path_length} bước")
        )

def run_app():
    root = tk.Tk()
    app = VacuumSimulatorApp(root)
    root.mainloop()

if __name__ == "__main__":
    run_app()
